In [27]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [28]:
df = pd.read_csv('/content/iris.csv')
print(df.head())

   sepal_length  sepal_width  petal_length  petal_width species
0           5.1          3.5           1.4          0.2  setosa
1           4.9          3.0           1.4          0.2  setosa
2           4.7          3.2           1.3          0.2  setosa
3           4.6          3.1           1.5          0.2  setosa
4           5.0          3.6           1.4          0.2  setosa


In [29]:
#assuming last column is target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]


In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [31]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [32]:
models = {
    'logistic regression': LogisticRegression(max_iter=1000),
    'random forest': RandomForestClassifier(),
    'svm': SVC()
}

In [33]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1 score': f1_score(y_test, y_pred, average='weighted')
    }

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    metrics = evaluate_model(model, X_test_scaled, y_test)
    results[name] = metrics

In [34]:
#inital results
print("\ninitial model evaluation:")
for model_name, scores in results.items():
    print(f"\n{model_name}:")
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")



initial model evaluation:

logistic regression:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1 score: 1.0000

random forest:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1 score: 1.0000

svm:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1 score: 1.0000


In [35]:
grid_params_svm = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

svm_grid = GridSearchCV(SVC(), grid_params_svm, cv=5, scoring='f1_weighted')
svm_grid.fit(X_train_scaled, y_train)
print("\nbest parameters for svm (gridsearchcv) are:", svm_grid.best_params_)


best parameters for svm (gridsearchcv) are: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}


In [37]:
random_params_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5, 10]
}
rf_random = RandomizedSearchCV(RandomForestClassifier(), random_params_rf, cv=5, scoring='f1_weighted', n_iter=10, random_state=42)
rf_random.fit(X_train_scaled, y_train)
print("\nbest parameters for random forest (randomizedsearchcv) are:", rf_random.best_params_)


best parameters for random forest (randomizedsearchcv) are: {'n_estimators': 200, 'min_samples_split': 10, 'max_depth': None}


In [38]:
tuned_models = {
    'svm (tuned)': svm_grid.best_estimator_,
    'random forest (tuned)': rf_random.best_estimator_
}

for name, model in tuned_models.items():
    model.fit(X_train_scaled, y_train)
    metrics = evaluate_model(model, X_test_scaled, y_test)
    results[name] = metrics

In [39]:
#final results
print("\nthis is my final model evaluation (after tuning):")
for model_name, scores in results.items():
    print(f"\n{model_name}:")
    for metric, value in scores.items():
        print(f"{metric}: {value:.4f}")


this is my final model evaluation (after tuning):

logistic regression:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1 score: 1.0000

random forest:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1 score: 1.0000

svm:
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1 score: 1.0000

svm (tuned):
accuracy: 0.9667
precision: 0.9694
recall: 0.9667
f1 score: 0.9664

random forest (tuned):
accuracy: 1.0000
precision: 1.0000
recall: 1.0000
f1 score: 1.0000
